In [ ]:
from IPython.display import display, HTML

html_content = """
<!DOCTYPE html>
<html>
<head>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(45deg, #ff7f50, #87cefa, #32cd32, #ff6347);
            background-size: 400% 400%;
            animation: gradientAnimation 10s ease infinite;
            color: #333;
            margin: 0;
            padding: 0;
            height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            flex-direction: column;
            overflow: hidden;
        }

        @keyframes gradientAnimation {
            0% { background-position: 0% 50%; }
            50% { background-position: 100% 50%; }
            100% { background-position: 0% 50%; }
        }

        .container {
            padding: 40px;
            text-align: center;
            background-color: rgba(255, 255, 255, 0.9);
            border-radius: 20px;
            box-shadow: 0px 10px 20px rgba(0, 0, 0, 0.1);
            margin-top: 50px;
            width: 70%;
            position: relative;
        }

        h1 {
            color: #2C3E50;
            font-size: 40px;
            background-color: #FF6347;
            padding: 20px;
            border-radius: 10px;
            margin-bottom: 20px;
            text-transform: uppercase;
        }

        h2 {
            color: #27AE60;
            font-size: 28px;
            background-color: #E8F8F5;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }

        .info {
            font-size: 20px;
            line-height: 1.8;
            color: #555;
            padding: 15px;
            background-color: #ECF0F1;
            border-radius: 10px;
            margin-top: 20px;
        }

        .footer {
            font-size: 16px;
            color: #7F8C8D;
            margin-top: 30px;
        }

        .footer a {
            color: #FF6347;
            text-decoration: none;
            font-weight: bold;
        }

        .scrolling-text {
            position: absolute;
            bottom: 10px;
            width: 100%;
            font-size: 24px;
            font-weight: bold;
            color: #FF6347;
            text-transform: uppercase;
            white-space: nowrap;
            overflow: hidden;
        }

        .scrolling-text span {
            display: inline-block;
            animation: scroll 15s linear infinite;
        }

        @keyframes scroll {
            from {
                transform: translateX(100%);
            }
            to {
                transform: translateX(-100%);
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Welcome to IS Project</h1>
        <h2>Phishing URL Detector</h2>
        <div class="info">
            <p><strong>Members:</strong></p>
            <p><strong>1. Muhammad Bilal Ashiq</strong></p>
            <p><strong>2. Faiez Tariq</strong></p>
            <p><strong>3. Bilal Ahmad </strong></p>
            <p><strong>4. Ali Hamza</strong></p>
            <p><strong>Degree Program: BS Computer Science</strong></p>
        </div>
        <div class="footer">
            <p>For further details, <a href="#">Contact Us</a></p>
        </div>
        <div class="scrolling-text">
            <span>Welcome to the IS -  Project. Get ready for an exciting journey into the world of Information Security!</span>
        </div>
    </div>
</body>
</html>
"""

# Display the HTML content
display(HTML(html_content))


# Libraries and Setup

In [ ]:
!pip install protobuf==3.20.3

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Dataset Preparation

In [ ]:
DATA_FILE = "/kaggle/input/malicious-urls-dataset/malicious_phish.csv"
SAMPLES_PER_CLASS = 50000
MAX_LEN = 128
BATCH_SIZE = 32

df = pd.read_csv(DATA_FILE)
df = df[df['type'].isin(['benign', 'defacement', 'phishing'])]
df = df.groupby('type').apply(lambda x: x.sample(SAMPLES_PER_CLASS)).reset_index(drop=True)

train_df, val_df = train_test_split(df, test_size=0.1, stratify=df['type'])

print(f"Dataset Prepared: {len(df)} total entries.")
print(f"Training: {len(train_df)} | Validation: {len(val_df)}")

# Tokenizer Setup

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.add_special_tokens({'pad_token': '<|pad|>', 'additional_special_tokens': ['<|url|>', '<|class|>', '<|end|>']})

class URLDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = f"<|url|>\n{row['url']}\n<|class|>\n{row['type']}<|end|>"
        enc = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)

train_loader = DataLoader(URLDataset(train_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(URLDataset(val_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

# Model Setup

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))
model.config.attn_pdrop = 0.2
model.config.resid_pdrop = 0.2
model.config.embd_pdrop = 0.2
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
epochs = 2

# Optimizer Setup

In [ ]:
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1} Train")
    
    for ids, mask in progress:
        ids, mask = ids.to(device), mask.to(device)
        outputs = model(ids, attention_mask=mask, labels=ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_train_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for ids, mask in val_loader:
            ids, mask = ids.to(device), mask.to(device)
            outputs = model(ids, attention_mask=mask, labels=ids)
            total_val_loss += outputs.loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

In [ ]:
model.save_pretrained("gpt2_phishing_30k")
tokenizer.save_pretrained("gpt2_phishing_30k")
print("Training Complete & Model Saved!")

In [ ]:
def predict(url):
    model.eval()
    prompt = f"<|url|>\n{url}\n<|class|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(max_new_tokens=6, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.encode("<|end|>")[0], **inputs)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    try:
        return decoded.split("<|class|>\n")[1].split("<|end|>")[0]
    except:
        return "Error"

print("google.com ->", predict("http://google.com"))
print("evil-site.net ->", predict("http://evil-site.net/login.php"))

# Evaluation on Validation Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
model.eval()

y_true, y_pred = [], []
batch_size = 32
val_records = val_df.to_dict('records')

with torch.no_grad():
    for i in range(0, len(val_records), batch_size):
        batch = val_records[i:i+batch_size]
        batch_prompts = [f"<|url|>\n{r['url']}\n<|class|>\n" for r in batch]
        true_labels = [r['type'] for r in batch]
        inputs = tokenizer(batch_prompts, padding=True, truncation=True, return_tensors="pt").to(device)
        outputs = model.generate(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], max_new_tokens=5, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.encode("<|end|>")[0])
        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        for idx, text in enumerate(decoded_outputs):
            try:
                prediction = text.split("<|class|>\n")[-1].replace("<|end|>", "").strip()
            except:
                prediction = "error"
            y_pred.append(prediction)
            y_true.append(true_labels[idx])
            if i==0 and idx<3:
                print(f"Target: {true_labels[idx]} | Pred: {prediction}")

labels = ['benign', 'defacement', 'phishing']
print("\nFINAL CLASSIFICATION REPORT")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()